In [1]:
import numpy
import stable_baselines3
import torch
import gymnasium as gym
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

### 1) Train a policy for Task 1

In [2]:
import math
import time
import random
import numpy as np
import torch
import gymnasium as gym
from dataclasses import dataclass

# ppo_scratch.py
# Minimal PPO training where the actor is an instance of nn.Sequential (Discrete actions)

import torch.nn.functional as F

@dataclass
class PPOConfig:
    env_id: str = "CartPole-v1"
    seed: int = 42
    total_timesteps: int = 100_000
    rollout_steps: int = 2048
    update_epochs: int = 10
    minibatch_size: int = 64
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    lr: float = 3e-4
    max_grad_norm: float = 0.5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

def make_actor_critic(obs_dim: int, n_actions: int):
    # Actor is an nn.Sequential as requested
    actor = torch.nn.Sequential(
        torch.nn.Linear(obs_dim, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, n_actions),
    )
    critic = torch.nn.Sequential(
        torch.nn.Linear(obs_dim, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 1),
    )
    return actor, critic

def set_seed(env, seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        env.reset(seed=seed)
    except TypeError:
        env.reset()
        env.action_space.seed(seed)

def evaluate(env_id, actor, device, episodes=10):
    actor.eval()
    scores = []
    for _ in range(episodes):
        env = gym.make(env_id)
        obs, _ = env.reset(seed=np.random.randint(0, 10_000_000))
        ep_r = 0.0
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                logits = actor(obs_t)
                action = torch.argmax(logits, dim=-1).item()
            obs, r, terminated, truncated, _ = env.step(action)
            ep_r += r
            done = terminated or truncated
        env.close()
        scores.append(ep_r)
    actor.train()
    return float(np.mean(scores)), float(np.std(scores))

def ppo_train(cfg: PPOConfig):
    env = gym.make(cfg.env_id)
    set_seed(env, cfg.seed)

    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    actor, critic = make_actor_critic(obs_dim, n_actions)
    assert isinstance(actor, torch.nn.Sequential), "Actor must be nn.Sequential"
    device = torch.device(cfg.device)
    actor.to(device)
    critic.to(device)

    optimizer = torch.optim.Adam([
        {"params": actor.parameters(), "lr": cfg.lr},
        {"params": critic.parameters(), "lr": cfg.lr},
    ])

    obs, _ = env.reset(seed=cfg.seed)
    global_step = 0
    start_time = time.time()

    while global_step < cfg.total_timesteps:
        # Storage for rollout
        obs_buf = np.zeros((cfg.rollout_steps, obs_dim), dtype=np.float32)
        act_buf = np.zeros((cfg.rollout_steps,), dtype=np.int64)
        logp_buf = np.zeros((cfg.rollout_steps,), dtype=np.float32)
        rew_buf = np.zeros((cfg.rollout_steps,), dtype=np.float32)
        done_buf = np.zeros((cfg.rollout_steps,), dtype=np.float32)
        val_buf = np.zeros((cfg.rollout_steps,), dtype=np.float32)

        # Collect rollout
        for t in range(cfg.rollout_steps):
            obs_buf[t] = obs
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                logits = actor(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                value = critic(obs_t).squeeze(-1)
                action = dist.sample()
                logp = dist.log_prob(action)
            act = int(action.item())
            next_obs, reward, terminated, truncated, _ = env.step(act)

            act_buf[t] = act
            logp_buf[t] = float(logp.item())
            rew_buf[t] = float(reward)
            done = terminated or truncated
            done_buf[t] = float(done)
            val_buf[t] = float(value.item())

            obs = next_obs
            global_step += 1
            if done:
                obs, _ = env.reset()

            if global_step >= cfg.total_timesteps:
                # If we hit total_timesteps mid-rollout, cut here
                obs_buf = obs_buf[:t+1]
                act_buf = act_buf[:t+1]
                logp_buf = logp_buf[:t+1]
                rew_buf = rew_buf[:t+1]
                done_buf = done_buf[:t+1]
                val_buf = val_buf[:t+1]
                break

        # Bootstrap with value of last observation
        with torch.no_grad():
            last_val = critic(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)).item()

        # Compute GAE advantages and returns
        T = len(rew_buf)
        adv_buf = np.zeros_like(rew_buf)
        last_gae = 0.0
        for t in reversed(range(T)):
            next_nonterminal = 1.0 - done_buf[t]
            next_value = last_val if t == T - 1 else val_buf[t + 1]
            delta = rew_buf[t] + cfg.gamma * next_value * next_nonterminal - val_buf[t]
            last_gae = delta + cfg.gamma * cfg.gae_lambda * next_nonterminal * last_gae
            adv_buf[t] = last_gae
        ret_buf = adv_buf + val_buf

        # Normalize advantages
        adv_t = torch.tensor(adv_buf, dtype=torch.float32, device=device)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std(unbiased=False) + 1e-8)

        obs_t = torch.tensor(obs_buf, dtype=torch.float32, device=device)
        act_t = torch.tensor(act_buf, dtype=torch.int64, device=device)
        old_logp_t = torch.tensor(logp_buf, dtype=torch.float32, device=device)
        ret_t = torch.tensor(ret_buf, dtype=torch.float32, device=device)

        # PPO updates
        batch_size = T
        idxs = np.arange(batch_size)
        for _ in range(cfg.update_epochs):
            np.random.shuffle(idxs)
            for start in range(0, batch_size, cfg.minibatch_size):
                end = start + cfg.minibatch_size
                mb_idx = idxs[start:end]
                mb_obs = obs_t[mb_idx]
                mb_act = act_t[mb_idx]
                mb_old_logp = old_logp_t[mb_idx]
                mb_adv = adv_t[mb_idx]
                mb_ret = ret_t[mb_idx]

                logits = actor(mb_obs)
                dist = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(mb_act)
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - mb_old_logp)
                pg_loss1 = -mb_adv * ratio
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1.0 - cfg.clip_coef, 1.0 + cfg.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                v = critic(mb_obs).squeeze(-1)
                v_loss = F.mse_loss(v, mb_ret)

                loss = pg_loss + cfg.vf_coef * v_loss - cfg.ent_coef * entropy

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), cfg.max_grad_norm)
                optimizer.step()

        if global_step % (10 * cfg.rollout_steps) < cfg.rollout_steps:
            mean_r, std_r = evaluate(cfg.env_id, actor, device, episodes=10)
            elapsed = time.time() - start_time
            print(f"Steps={global_step} | meanR={mean_r:.1f} +/- {std_r:.1f} | elapsed={elapsed:.1f}s")

    env.close()
    # Final checks and evaluation
    print(f"Actor is nn.Sequential: {isinstance(actor, torch.nn.Sequential)}")
    mean_r, std_r = evaluate(cfg.env_id, actor, device, episodes=20)
    print(f"Final evaluation over 20 episodes: mean_reward={mean_r:.2f} +/- {std_r:.2f}")
    return actor, critic

### TRAINING
cfg = PPOConfig()
ppo_actor, ppo_critic = ppo_train(cfg)

Steps=20480 | meanR=450.0 +/- 56.1 | elapsed=4.4s
Steps=40960 | meanR=499.1 +/- 2.7 | elapsed=8.8s
Steps=61440 | meanR=500.0 +/- 0.0 | elapsed=13.1s
Steps=81920 | meanR=500.0 +/- 0.0 | elapsed=17.4s
Actor is nn.Sequential: True
Final evaluation over 20 episodes: mean_reward=500.00 +/- 0.00


In [3]:
# # Train a policy network for CartPole-v1 using Stable-Baselines3 PPO
# # Configure the actor to be an nn.Sequential via policy_kwargs (pi trunk)

# # Create vectorized environment
# env = make_vec_env("CartPole-v1", n_envs=1, seed=42)

# # Define custom MLP architecture; MlpExtractor builds pi/vf as nn.Sequential
# policy_kwargs = dict(
#     activation_fn=torch.nn.ReLU,
#     net_arch=dict(pi=[64, 64], vf=[64, 64]),
# )

# # Define and train the model
# model = stable_baselines3.PPO(
#     policy="MlpPolicy",
#     env=env,
#     policy_kwargs=policy_kwargs,
#     verbose=1,
#     seed=42,
#     device="auto",
# )

# model.learn(total_timesteps=100_000)

# # Verify actor is an nn.Sequential (the policy trunk)
# is_seq = isinstance(model.policy.mlp_extractor.policy_net, torch.nn.Sequential)
# print(f"Actor (policy_net) is nn.Sequential: {is_seq}")

# # Evaluate the trained policy
# mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)
# print(f"Evaluation over 10 episodes: mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

In [4]:
import torch.nn as nn
isinstance(ppo_actor, nn.Sequential)

True

### 2) Train a safety critic

#### a) Collect data for safety critic

In [5]:
# Train a safety critic that estimates probability of failure (termination) under the learned policy

device = 'cpu' # model.device
env_id = "CartPole-v1"
env = gym.make(env_id)
obs_dim = env.observation_space.shape[0]
action_dim = 1 if isinstance(env.action_space, gym.spaces.Discrete) else env.action_space.shape[0]

class SafetyCritic(torch.nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),  # logits for failure probability
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def collect_safety_dataset(policy_model, env_id, num_episodes=300, epsilon=0.2):
    eval_env = gym.make(env_id)
    max_steps = getattr(eval_env.spec, "max_episode_steps", 1000)
    obs_list, action_list, failure_list = [], [], []

    for _ in range(num_episodes):
        obs, _ = eval_env.reset()
        ep_obs = []
        ep_actions = []
        for _t in range(max_steps):
            ep_obs.append(obs.copy())

            if numpy.random.rand() < epsilon:
                action = eval_env.action_space.sample()
            else:
                ### For SB3 model
                # action, _ = policy_model.predict(obs, deterministic=True)
                ### For my PPO from scratch
                cur_obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
                logits = policy_model.forward(cur_obs_tensor)
                action = logits.argmax(dim=-1).item()
                if isinstance(action, numpy.ndarray) and action.shape == (1,):
                    action = action.item()
            ep_actions.append(action)

            obs, _r, terminated, truncated, _info = eval_env.step(action)
            if terminated or truncated:
                fail = 1.0 if (terminated and not truncated) else 0.0
                obs_list.extend(ep_obs)
                action_list.extend(ep_actions)
                failure_list.extend([fail] * len(ep_obs))
                break

    eval_env.close()
    obs_arr = numpy.asarray(obs_list, dtype=numpy.float32)
    action_arr = numpy.asarray(action_list, dtype=numpy.float32)
    failure_arr = numpy.asarray(failure_list, dtype=numpy.float32)
    return obs_arr, action_arr, failure_arr

# Collect rollouts with slight exploration to see failures
obs_arr, action_arr, failure_arr = collect_safety_dataset(ppo_actor, env_id=env_id, num_episodes=400, epsilon=0.25)
print(f"Safety dataset: {obs_arr.shape[0]} samples, positive (fail) rate={failure_arr.mean():.3f}")

Safety dataset: 192055 samples, positive (fail) rate=0.034


#### b) Train the safety critic using Monte-Carlo and TD targets

In [7]:
# Train a safety critic as a Q-function for failure probability using TD (Bellman) targets.
# 1) Supervised warm-start on (X, y) Monte Carlo labels.
# 2) TD fine-tuning on transitions collected from the fixed policy.

# Hyperparameters
gamma = 0.99
batch_size = 4096
warmup_epochs = 1
td_epochs = 100
lr = 1e-3
epsilon_collect = 0.2
num_episodes_collect = 400

# Instantiate safety critic
safety_critic = SafetyCritic(obs_dim+action_dim).to(device)
optimizer = torch.optim.Adam(safety_critic.parameters(), lr=lr)
bce_logits = torch.nn.BCEWithLogitsLoss()

# Warm-start on the collected (state, failure_label) dataset
X = numpy.concatenate([obs_arr, action_arr.reshape(-1, 1)], axis=1) # include action as input
y = failure_arr # failure labels
X_t = torch.from_numpy(X).to(device)
y_t = torch.from_numpy(y).to(device)

safety_critic.train()
N = X_t.shape[0]
for epoch in range(warmup_epochs):
    idx = numpy.random.permutation(N)
    total_loss = 0.0
    for i in range(0, N, batch_size):
        j = idx[i:i + batch_size]
        xb = X_t[j]
        yb = y_t[j]

        logits = safety_critic(xb)
        loss = bce_logits(logits, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(safety_critic.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"[Warmup {epoch+1}/{warmup_epochs}] BCE loss={total_loss / N:.6f}")

# Collect transitions for TD training
def collect_transitions(policy_model, env_id, num_episodes=300, epsilon=0.1):
    eval_env = gym.make(env_id)
    max_steps = getattr(eval_env.spec, "max_episode_steps", 1000)
    s_list, action_list, next_s_list, done_list, fail_list = [], [], [], [], []

    for _ in range(num_episodes):
        s, _ = eval_env.reset()
        for _t in range(max_steps):
            if numpy.random.rand() < epsilon:
                a = eval_env.action_space.sample()
            else:
                ### For SB3 model
                # a, _ = policy_model.predict(s, deterministic=True)
                ### For my PPO from scratch
                cur_s_tensor = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                logits = policy_model.forward(cur_s_tensor)
                a = logits.argmax(dim=-1).item()
                if isinstance(a, numpy.ndarray) and a.shape == (1,):
                    a = a.item()

            next_s, _r, terminated, truncated, _info = eval_env.step(a)
            done = bool(terminated or truncated)
            fail = 1.0 if (terminated and not truncated) else 0.0

            s_list.append(s.copy())
            action_list.append(a)
            next_s_list.append(next_s.copy())
            done_list.append(float(done))
            fail_list.append(float(fail))

            s = next_s
            if done:
                break

    eval_env.close()
    S = numpy.asarray(s_list, dtype=numpy.float32)
    A = numpy.asarray(action_list, dtype=numpy.float32)
    SP = numpy.asarray(next_s_list, dtype=numpy.float32)
    D = numpy.asarray(done_list, dtype=numpy.float32)
    F = numpy.asarray(fail_list, dtype=numpy.float32)
    return S, A, SP, D, F

S, A, SP, D, Failures = collect_transitions(ppo_actor, env_id, num_episodes=num_episodes_collect, epsilon=epsilon_collect)
print(f"TD dataset: {S.shape[0]} transitions, fail terminals={Failures.sum():.0f}")

# TD training
S_t = torch.from_numpy(S).to(device)
A_t = torch.from_numpy(A).to(device)
SP_t = torch.from_numpy(SP).to(device)
D_t = torch.from_numpy(D).to(device)
F_t = torch.from_numpy(Failures).to(device)

M = S_t.shape[0]
for epoch in range(td_epochs):
    idx = numpy.random.permutation(M)
    total_loss = 0.0
    for i in range(0, M, batch_size):
        j = idx[i:i + batch_size]
        s_b = S_t[j]
        a_b = A_t[j].unsqueeze(-1)
        state_action_b = torch.cat([s_b, a_b], dim=1)
        d_b = D_t[j]
        f_b = F_t[j]

        sp_b = SP_t[j]
        # Apply the policy to all next states
        if isinstance(env.action_space, gym.spaces.Discrete):
            with torch.no_grad():
                ### For SB3 model
                # ap_b_np, _ = model.predict(sp_b.cpu().numpy(), deterministic=True)
                ### For my PPO from scratch
                cur_sp_tensor = torch.tensor(sp_b, dtype=torch.float32).unsqueeze(0).to(device)
                logits = ppo_actor.forward(cur_sp_tensor)
                ap_b_np = logits.argmax(dim=-1).cpu().numpy()
            ap_b = torch.from_numpy(ap_b_np).to(device).unsqueeze(-1)
        else:
            with torch.no_grad():
                ### For SB3 model
                # ap_b_np, _ = model.predict(sp_b.cpu().numpy(), deterministic=True)
                ### For my PPO from scratch
                cur_sp_tensor = torch.tensor(sp_b, dtype=torch.float32).unsqueeze(0).to(device)
                logits = ppo_actor.forward(cur_sp_tensor)
                ap_b_np = logits.argmax(dim=-1).cpu().numpy()
            ap_b = torch.from_numpy(ap_b_np).to(device)
        if len(ap_b.shape) > 2:
            ap_b = ap_b.reshape(-1, 1)
        next_state_action_b = torch.cat([sp_b, ap_b], dim=1)

        with torch.no_grad():
            next_logits = safety_critic(next_state_action_b)
            next_p = torch.sigmoid(next_logits)
            target = f_b + (1.0 - d_b) * gamma * next_p
            target = target.clamp(0.0, 1.0)

        logits = safety_critic(state_action_b)
        loss = bce_logits(logits, target)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(safety_critic.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * s_b.size(0)

    print(f"[TD {epoch+1}/{td_epochs}] BCE(TD) loss={total_loss / M:.6f}")

safety_critic.eval()

# Quick sanity check: estimated failure rate on a batch of states
with torch.no_grad():
    est_fail_rate = torch.sigmoid(safety_critic(X_t[:10000])).mean().item()
print(f"Estimated mean failure probability on sample states: {est_fail_rate:.4f}")

[Warmup 1/1] BCE loss=0.455697
TD dataset: 196621 transitions, fail terminals=12
[TD 1/100] BCE(TD) loss=0.315639
[TD 2/100] BCE(TD) loss=0.310236


/var/folders/29/nv2h4vrx5n9791kr6gs7kmfw0000gp/T/ipykernel_18572/517827324.py:117: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  cur_sp_tensor = torch.tensor(sp_b, dtype=torch.float32).unsqueeze(0).to(device)


[TD 3/100] BCE(TD) loss=0.307282
[TD 4/100] BCE(TD) loss=0.300792
[TD 5/100] BCE(TD) loss=0.293888
[TD 6/100] BCE(TD) loss=0.281931
[TD 7/100] BCE(TD) loss=0.267030
[TD 8/100] BCE(TD) loss=0.249363
[TD 9/100] BCE(TD) loss=0.225014
[TD 10/100] BCE(TD) loss=0.194509
[TD 11/100] BCE(TD) loss=0.160211
[TD 12/100] BCE(TD) loss=0.124419
[TD 13/100] BCE(TD) loss=0.100077
[TD 14/100] BCE(TD) loss=0.080234
[TD 15/100] BCE(TD) loss=0.065351
[TD 16/100] BCE(TD) loss=0.095569
[TD 17/100] BCE(TD) loss=0.093451
[TD 18/100] BCE(TD) loss=0.081388
[TD 19/100] BCE(TD) loss=0.073231
[TD 20/100] BCE(TD) loss=0.067144
[TD 21/100] BCE(TD) loss=0.062295
[TD 22/100] BCE(TD) loss=0.058411
[TD 23/100] BCE(TD) loss=0.054111
[TD 24/100] BCE(TD) loss=0.051008
[TD 25/100] BCE(TD) loss=0.048516
[TD 26/100] BCE(TD) loss=0.045881
[TD 27/100] BCE(TD) loss=0.043545
[TD 28/100] BCE(TD) loss=0.041513
[TD 29/100] BCE(TD) loss=0.039086
[TD 30/100] BCE(TD) loss=0.037527
[TD 31/100] BCE(TD) loss=0.036022
[TD 32/100] BCE(TD) l

#### 4) Dataset labelled by the safety critic
- Collect as many unique observations as possible
- For each unique observation, query the safety critic to calculate failure probability for each possible action

In [8]:
# Collect a large exploratory CartPole dataset using the trained policy with epsilon-greedy exploration

episodes_exp = 1500   # increase for more data
epsilon_exp = 0.5    # higher -> more random actions

S_exp, A_exp, SP_exp, D_exp, F_exp = collect_transitions(
    ppo_actor, env_id, num_episodes=episodes_exp, epsilon=epsilon_exp
)

N = S_exp.shape[0]
avg_len = N / episodes_exp
fail_terms = int(F_exp.sum())

print(f"Exploration dataset collected:")
print(f"- episodes={episodes_exp}, epsilon={epsilon_exp}")
print(f"- transitions={N}, mean_ep_len≈{avg_len:.1f}, fail terminals={fail_terms}")

# Optional: estimate number of unique observations (rounded to reduce near-duplicates)
rounded_S = numpy.round(S_exp, 3)
unique_obs = numpy.unique(rounded_S, axis=0)
print(f"- approx unique observations (rounded 3 d.p.)={unique_obs.shape[0]}")

# Optional: save to disk for reuse
# numpy.savez_compressed(
#     "cartpole_exploration_dataset.npz",
#     S=S_exp, A=A_exp, SP=SP_exp, D=D_exp, F=F_exp, epsilon=epsilon_exp
# )
# print("Saved to cartpole_exploration_dataset.npz")

Exploration dataset collected:
- episodes=1500, epsilon=0.5
- transitions=251748, mean_ep_len≈167.8, fail terminals=1436
- approx unique observations (rounded 3 d.p.)=251746


In [9]:
# For each state in S_exp, find all discrete actions with failure probability < 0.5 (per safety_critic)

# Determine the number of discrete actions
try:
    n_actions = env.action_space.n
except Exception:
    n_actions = gym.make(env_id).action_space.n  # fallback

safety_critic.eval()
threshold = 0.5
N = S_exp.shape[0]
batch_size_eval = 32768  # adjust if needed

safe_action_mask = numpy.zeros((N, n_actions), dtype=bool)
safe_actions_per_state = []

with torch.no_grad():
    for i in range(0, N, batch_size_eval):
        j = min(i + batch_size_eval, N)
        s_batch = torch.from_numpy(S_exp[i:j]).to(device=device, dtype=torch.float32)

        # Build (state, action) pairs for all discrete actions
        action_vals = torch.arange(n_actions, device=device, dtype=s_batch.dtype)  # [0..n_actions-1]
        s_expanded = s_batch.unsqueeze(1).expand(-1, n_actions, -1)                # (B, A, obs_dim)
        a_expanded = action_vals.view(1, n_actions, 1).expand(s_batch.size(0), n_actions, 1)  # (B, A, 1)
        sa_flat = torch.cat([s_expanded, a_expanded], dim=2).reshape(-1, obs_dim + 1)        # (B*A, obs_dim+1)

        logits = safety_critic(sa_flat)                         # (B*A,)
        probs = torch.sigmoid(logits).reshape(-1, n_actions)    # (B, A)
        mask = (probs < threshold).cpu().numpy()                # (B, A) bool

        safe_action_mask[i:j] = mask
        safe_actions_per_state.extend([numpy.where(row)[0].tolist() for row in mask])

print(f"Computed safe actions for {N} states (threshold={threshold}, n_actions={n_actions}).")
# Example: first 5 states
for k in range(min(5, N)):
    print(f"state[{k}] safe actions: {safe_actions_per_state[k]}")

Computed safe actions for 251748 states (threshold=0.5, n_actions=2).
state[0] safe actions: [0, 1]
state[1] safe actions: [0, 1]
state[2] safe actions: [0, 1]
state[3] safe actions: [0, 1]
state[4] safe actions: [0, 1]


In [10]:
import pandas as pd
pd.Series([len(lst) for lst in safe_actions_per_state]).value_counts().sort_index()

0       675
1      1078
2    249995
Name: count, dtype: int64

In [11]:
num_actions = env.action_space.n if isinstance(env.action_space, gym.spaces.Discrete) else env.action_space.shape[0]

new_safe_actions_per_state = []
for safe_action_indices in safe_actions_per_state:
    safe_action_indices = torch.tensor(safe_action_indices, dtype=torch.long)
    pad_len = max(0, num_actions - safe_action_indices.numel())
    if pad_len > 0:
        pad = torch.full((pad_len,), -1, dtype=safe_action_indices.dtype, device=safe_action_indices.device)
        cur_tensor = torch.cat([safe_action_indices, pad], dim=0)
    else:
        cur_tensor = safe_action_indices
    new_safe_actions_per_state.append(cur_tensor.numpy())

### Rashomon set calculation

In [12]:
from torch.utils.data import TensorDataset, DataLoader

In [13]:
safe_actions_tensor = torch.from_numpy(np.array(new_safe_actions_per_state))
states_tensor = torch.from_numpy(S_exp)
state_action_torch_dataset = TensorDataset(states_tensor, safe_actions_tensor)
state_action_loader = DataLoader(state_action_torch_dataset, batch_size=8, shuffle=False)

In [14]:
import os
import sys

project_root = os.path.abspath('/Users/ma5923/Documents/_projects/CertifiedContinualLearning')
if project_root not in sys.path:
    sys.path.append(project_root)

In [15]:
from scripts._sqrl_pretrain import SQRLPretrainConfig, pretrain_sqrl, default_failure_fn
from scripts.custom_tasks import *
from src.trainer import IntervalTrainer
from torch.utils.data import TensorDataset, DataLoader
from stable_baselines3 import PPO

In [16]:
# Use IntervalTrainer to compute the Rashomon set around the pretrained policy
interval_trainer = IntervalTrainer(
    model=ppo_actor, # policy which is an instance of nn.Sequential
    # min_acc_limit=interval_trainer_min_accuracy,
    seed=2025,
    # TODO: set accuracy?
)
interval_trainer.compute_rashomon_set(
    dataset=state_action_torch_dataset, # states and safe actions
    multi_label=True # NOTE
)

Initial acc constraint violation: -0.0961 (Positive = violated)
Number of model parameters: 4610
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|██████████| 2000/2000 [00:44<00:00, 44.60it/s, size=7911.76, obj=1.715, min_soft_acc=0.998]


Final bbox:  Obj=1.72,  Size=7911.76,  Min acc hard=1.00,  Min acc soft=1.00
Computing final certificates over 256 samples
Num cert samples: 256
----------------------- Finished Computing Rashomon set ------------------------


In [17]:
# Store parameter bounds and certificates
print("Rashomon set computation completed!")
print(f"Number of bounded models: {len(interval_trainer.bounds)}")
print(f"Number of certificates: {len(interval_trainer.certificates)}")

# Extract parameter bounds from the bounded model
assert len(interval_trainer.bounds) == 1, "Expected exactly one bounded model"
bounded_model = interval_trainer.bounds[0]
param_bounds_l = [bound.detach().cpu() for bound in bounded_model.param_l]
param_bounds_u = [bound.detach().cpu() for bound in bounded_model.param_u]

print(f"\nParameter bounds information:")
print(f"Number of parameter tensors: {len(param_bounds_l)}")

total_params = 0
for i, (p_l, p_u) in enumerate(zip(param_bounds_l, param_bounds_u)):
    width = (p_u - p_l).abs().mean().item()
    total_params += p_l.numel()
    print(f"  Parameter {i}: shape={p_l.shape}, avg_width={width:.6f}, params={p_l.numel()}")
print(f"Total parameters: {total_params}")

# Certificate information
assert len(interval_trainer.certificates) == 1, "Expected exactly one certificate"
certificate = interval_trainer.certificates[0]
print(f"Certified accuracy on the safe action dataset: {certificate:.2f}")

Rashomon set computation completed!
Number of bounded models: 1
Number of certificates: 1

Parameter bounds information:
Number of parameter tensors: 6
  Parameter 0: shape=torch.Size([64, 4]), avg_width=1.716210, params=256
  Parameter 1: shape=torch.Size([64]), avg_width=1.716211, params=64
  Parameter 2: shape=torch.Size([64, 64]), avg_width=1.716218, params=4096
  Parameter 3: shape=torch.Size([64]), avg_width=1.716219, params=64
  Parameter 4: shape=torch.Size([2, 64]), avg_width=1.716215, params=128
  Parameter 5: shape=torch.Size([2]), avg_width=1.716220, params=2
Total parameters: 4610
Certified accuracy on the safe action dataset: 1.00


### Task 2

In [18]:
from gymnasium.envs.registration import register
from gymnasium.envs.classic_control.cartpole import CartPoleEnv

class CartPoleDiff(CartPoleEnv):
    """
    CartPole with slightly different dynamics and stricter termination:
    - Heavier pole and cart, slightly longer pole, slightly weaker force
    - Slightly stronger gravity
    - Smaller angle/x thresholds
    - Additional failure if |x_dot| or |theta_dot| exceed thresholds
    """
    def __init__(
        self,
        render_mode=None,
        gravity_scale=1.07,
        masscart=1.2,
        masspole=100, # NOTE: much heavier pole
        length=0.55,
        force_mag=9.0,
        theta_threshold_deg=10.0,
        x_threshold=2.0,
        xdot_fail_thresh=2.5,
        thetadot_fail_thresh=3.5,
        warmup_steps=5,
    ):
        super().__init__(render_mode=render_mode)
        # Adjust dynamics
        self.gravity *= gravity_scale
        self.masscart = float(masscart)
        self.masspole = float(masspole)
        self.total_mass = self.masspole + self.masscart
        self.length = float(length)
        self.polemass_length = self.masspole * self.length
        self.force_mag = float(force_mag)

        # Adjust termination thresholds
        self.theta_threshold_radians = math.radians(theta_threshold_deg)
        self.x_threshold = float(x_threshold)

        # Extra failure logic
        self.xdot_fail_thresh = float(xdot_fail_thresh)
        self.thetadot_fail_thresh = float(thetadot_fail_thresh)
        self._warmup_steps = int(warmup_steps)
        self._elapsed_steps = 0

    def reset(self, *, seed=None, options=None):
        self._elapsed_steps = 0
        return super().reset(seed=seed, options=options)

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        self._elapsed_steps += 1

        # Additional failure on high velocities after a few warmup steps
        x, x_dot, theta, theta_dot = obs
        high_speed_fail = (
            (self._elapsed_steps > self._warmup_steps) and
            (abs(x_dot) > self.xdot_fail_thresh or abs(theta_dot) > self.thetadot_fail_thresh)
        )

        if high_speed_fail:
            terminated = True
            info = dict(info)
            info["custom_termination"] = "high_speed_fail"

        return obs, reward, terminated, truncated, info

# Register with a longer time limit (truncation) to distinguish from failure termination
env_id_diff = "CartPoleDiff-v1"
register(
    id=env_id_diff,
    entry_point=CartPoleDiff,  # callable is supported by Gymnasium
    max_episode_steps=600,
)

# Quick smoke test (avoid clobbering existing `env`)
_env = gym.make(env_id_diff)
obs, _ = _env.reset(seed=getattr(cfg, "seed", 42))
for _ in range(5):
    obs, _, terminated, truncated, _ = _env.step(_env.action_space.sample())
    if terminated or truncated:
        break
_env.close()
print(f"Registered {env_id_diff} with altered dynamics and termination logic.")

Registered CartPoleDiff-v1 with altered dynamics and termination logic.


### Adaptation without the Rashomon set bounds

In [22]:
import torch.nn.functional as F
import copy

# Fine-tune PPO on CartPoleDiff starting from ppo_actor/ppo_critic without Rashomon constraints

def ppo_train_cartpole_diff_free(
    warm_actor: torch.nn.Module,
    warm_critic: torch.nn.Module,
    cfg_obj,
    env_id_str: str,
):
    env = gym.make(env_id_str)
    set_seed(env, cfg_obj.seed)

    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    assert isinstance(warm_actor, torch.nn.Sequential)
    assert warm_actor[-1].out_features == n_actions, "Actor output dim must match env action space"

    actor = warm_actor.__class__(*warm_actor.children()) if False else copy.deepcopy(warm_actor)
    critic = copy.deepcopy(warm_critic)
    actor.to(cfg_obj.device)
    critic.to(cfg_obj.device)

    optimizer = torch.optim.Adam(
        [{"params": actor.parameters(), "lr": cfg_obj.lr},
         {"params": critic.parameters(), "lr": cfg_obj.lr}]
    )

    obs, _ = env.reset(seed=cfg_obj.seed)
    global_step = 0
    start_time = time.time()

    while global_step < cfg_obj.total_timesteps:
        T = cfg_obj.rollout_steps
        obs_buf = np.zeros((T, obs_dim), dtype=np.float32)
        act_buf = np.zeros((T,), dtype=np.int64)
        logp_buf = np.zeros((T,), dtype=np.float32)
        rew_buf = np.zeros((T,), dtype=np.float32)
        done_buf = np.zeros((T,), dtype=np.float32)
        val_buf = np.zeros((T,), dtype=np.float32)

        for t in range(T):
            obs_buf[t] = obs
            obs_t = torch.tensor(obs, dtype=torch.float32, device=cfg_obj.device).unsqueeze(0)
            with torch.no_grad():
                logits = actor(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                value = critic(obs_t).squeeze(-1)
                action = dist.sample()
                logp = dist.log_prob(action)
            a = int(action.item())
            next_obs, reward, terminated, truncated, _ = env.step(a)

            act_buf[t] = a
            logp_buf[t] = float(logp.item())
            rew_buf[t] = float(reward)
            done = terminated or truncated
            done_buf[t] = float(done)
            val_buf[t] = float(value.item())

            obs = next_obs
            global_step += 1
            if done:
                obs, _ = env.reset()

            if global_step >= cfg_obj.total_timesteps:
                obs_buf = obs_buf[:t+1]
                act_buf = act_buf[:t+1]
                logp_buf = logp_buf[:t+1]
                rew_buf = rew_buf[:t+1]
                done_buf = done_buf[:t+1]
                val_buf = val_buf[:t+1]
                break

        with torch.no_grad():
            last_val = critic(torch.tensor(obs, dtype=torch.float32, device=cfg_obj.device).unsqueeze(0)).item()

        Tlen = len(rew_buf)
        adv_buf = np.zeros_like(rew_buf)
        last_gae = 0.0
        for t in reversed(range(Tlen)):
            next_nonterminal = 1.0 - done_buf[t]
            next_value = last_val if t == Tlen - 1 else val_buf[t + 1]
            delta = rew_buf[t] + cfg_obj.gamma * next_value * next_nonterminal - val_buf[t]
            last_gae = delta + cfg_obj.gamma * cfg_obj.gae_lambda * next_nonterminal * last_gae
            adv_buf[t] = last_gae
        ret_buf = adv_buf + val_buf

        adv_t = torch.tensor(adv_buf, dtype=torch.float32, device=cfg_obj.device)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std(unbiased=False) + 1e-8)
        obs_t = torch.tensor(obs_buf, dtype=torch.float32, device=cfg_obj.device)
        act_t = torch.tensor(act_buf, dtype=torch.int64, device=cfg_obj.device)
        old_logp_t = torch.tensor(logp_buf, dtype=torch.float32, device=cfg_obj.device)
        ret_t = torch.tensor(ret_buf, dtype=torch.float32, device=cfg_obj.device)

        batch_size = Tlen
        idxs = np.arange(batch_size)
        for _ in range(cfg_obj.update_epochs):
            np.random.shuffle(idxs)
            for start in range(0, batch_size, cfg_obj.minibatch_size):
                end = start + cfg_obj.minibatch_size
                mb_idx = idxs[start:end]
                mb_obs = obs_t[mb_idx]
                mb_act = act_t[mb_idx]
                mb_old_logp = old_logp_t[mb_idx]
                mb_adv = adv_t[mb_idx]
                mb_ret = ret_t[mb_idx]

                logits = actor(mb_obs)
                dist = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(mb_act)
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - mb_old_logp)
                pg_loss1 = -mb_adv * ratio
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1.0 - cfg_obj.clip_coef, 1.0 + cfg_obj.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                v = critic(mb_obs).squeeze(-1)
                v_loss = F.mse_loss(v, mb_ret)

                loss = pg_loss + cfg_obj.vf_coef * v_loss - cfg_obj.ent_coef * entropy

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(actor.parameters()) + list(critic.parameters()),
                    cfg_obj.max_grad_norm
                )
                optimizer.step()

        if global_step % (10 * cfg_obj.rollout_steps) < cfg_obj.rollout_steps:
            mean_r, std_r = evaluate(env_id_str, actor, torch.device(cfg_obj.device), episodes=10)
            elapsed = time.time() - start_time
            print(f"[CartPoleDiff Free] Steps={global_step} | meanR={mean_r:.1f} +/- {std_r:.1f} | elapsed={elapsed:.1f}s")

    env.close()
    # print(f"Actor is nn.Sequential: {isinstance(actor, torch.nn.Sequential)}")
    mean_r, std_r = evaluate(env_id_str, actor, torch.device(cfg_obj.device), episodes=20)
    print(f"[CartPoleDiff Free] Final eval: mean_reward={mean_r:.2f} +/- {std_r:.2f}")
    return actor, critic

# Train without Rashomon constraints
ppo_actor_diff_free, ppo_critic_diff_free = ppo_train_cartpole_diff_free(
    warm_actor=ppo_actor,
    warm_critic=ppo_critic,
    cfg_obj=cfg,
    env_id_str=env_id_diff,
)

[CartPoleDiff Free] Steps=20480 | meanR=197.8 +/- 263.4 | elapsed=4.7s
[CartPoleDiff Free] Steps=40960 | meanR=28.6 +/- 11.0 | elapsed=9.1s
[CartPoleDiff Free] Steps=61440 | meanR=111.8 +/- 176.0 | elapsed=13.5s
[CartPoleDiff Free] Steps=81920 | meanR=79.1 +/- 173.7 | elapsed=18.1s
[CartPoleDiff Free] Final eval: mean_reward=199.35 +/- 262.45


In [23]:
### Check safety on task 1
# Safety check of ppo_actor_diff on the original CartPole-v1 using empirical failures and the safety_critic

episodes_check = 200
max_steps_limit = getattr(gym.make(env_id).spec, "max_episode_steps", 500)

ppo_actor_diff_free.eval()
safety_critic.eval()

fail_episodes = 0
ep_lengths = []
mean_pred_fail_probs = []
max_pred_fail_probs = []
safe_step_count = 0
total_step_count = 0

eval_env = gym.make(env_id)

with torch.no_grad():
    for _ in range(episodes_check):
        obs, _ = eval_env.reset(seed=np.random.randint(0, 10_000_000))
        ep_probs = []
        steps = 0
        while True:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            logits = ppo_actor_diff_free(obs_t)
            action = torch.argmax(logits, dim=-1).item()

            # Predicted failure probability for (state, action)
            a_t = torch.tensor([[float(action)]], dtype=torch.float32, device=device)
            sa = torch.cat([obs_t, a_t], dim=1)
            pred_fail_prob = torch.sigmoid(safety_critic(sa)).item()
            ep_probs.append(pred_fail_prob)

            # Count "safe" steps per chosen action (threshold defined earlier)
            safe_step_count += 1 if pred_fail_prob < threshold else 0
            total_step_count += 1

            obs, _, terminated, truncated, info = eval_env.step(action)
            steps += 1

            if terminated or truncated or steps >= max_steps_limit:
                # Failure if terminated (not time-limit truncation)
                fail_episodes += 1 if (terminated and not truncated) else 0
                ep_lengths.append(steps)
                mean_pred_fail_probs.append(float(numpy.mean(ep_probs)) if ep_probs else 0.0)
                max_pred_fail_probs.append(float(numpy.max(ep_probs)) if ep_probs else 0.0)
                break

eval_env.close()

failure_rate = fail_episodes / episodes_check
mean_len = float(numpy.mean(ep_lengths)) if ep_lengths else 0.0
std_len = float(numpy.std(ep_lengths)) if ep_lengths else 0.0
avg_mean_pred_fail = float(numpy.mean(mean_pred_fail_probs)) if mean_pred_fail_probs else 0.0
avg_max_pred_fail = float(numpy.mean(max_pred_fail_probs)) if max_pred_fail_probs else 0.0
safe_step_ratio = (safe_step_count / max(1, total_step_count))

print("Safety check on CartPole-v1 with ppo_actor_diff_free:")
print(f"- episodes={episodes_check}, max_steps={max_steps_limit}")
print(f"- empirical failure rate (terminated only) = {failure_rate:.4f}")
print(f"- episode length: mean={mean_len:.1f} +/- {std_len:.1f}")
print(f"- safety_critic: mean predicted fail prob per episode = {avg_mean_pred_fail:.4e}")
print(f"- safety_critic: mean of episode-wise max fail prob   = {avg_max_pred_fail:.4e}")
print(f"- fraction of steps predicted safe (p_fail < {threshold}) = {safe_step_ratio:.4f}")


Safety check on CartPole-v1 with ppo_actor_diff_free:
- episodes=200, max_steps=500
- empirical failure rate (terminated only) = 0.1050
- episode length: mean=492.9 +/- 23.7
- safety_critic: mean predicted fail prob per episode = 1.1408e-02
- safety_critic: mean of episode-wise max fail prob   = 8.9702e-02
- fraction of steps predicted safe (p_fail < 0.5) = 0.9981


#### Adaptation with the Rashomon set bounds

In [24]:
import copy

# Transfer PPO training on CartPoleDiff with PGD projection into Rashomon bounds
def project_actor_(actor: torch.nn.Module, lower_bounds, upper_bounds):
    with torch.no_grad():
        for p, lb, ub in zip(actor.parameters(), lower_bounds, upper_bounds):
            assert p.shape == lb.shape == ub.shape, f"Shape mismatch: {p.shape} vs {lb.shape} vs {ub.shape}"
            p.data.clamp_(lb.to(p.device, dtype=p.dtype), ub.to(p.device, dtype=p.dtype))

def ppo_train_cartpole_diff_with_pgd(
    warm_actor: torch.nn.Module,
    warm_critic: torch.nn.Module,
    bounds_l,
    bounds_u,
    cfg_obj,
    env_id_str: str,
):
    env = gym.make(env_id_str)
    set_seed(env, cfg_obj.seed)

    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    assert isinstance(warm_actor, torch.nn.Sequential)
    assert warm_actor[-1].out_features == n_actions, "Actor output dim must match env action space"

    # Deepcopy warm start so original remains unchanged
    actor = copy.deepcopy(warm_actor).to(cfg_obj.device)
    critic = copy.deepcopy(warm_critic).to(cfg_obj.device)

    # Ensure initial parameters satisfy bounds
    project_actor_(actor, bounds_l, bounds_u)

    optimizer = torch.optim.Adam(
        [{"params": actor.parameters(), "lr": cfg_obj.lr},
         {"params": critic.parameters(), "lr": cfg_obj.lr}]
    )

    obs, _ = env.reset(seed=cfg_obj.seed)
    global_step = 0
    start_time = time.time()

    while global_step < cfg_obj.total_timesteps:
        # Rollout buffers
        T = cfg_obj.rollout_steps
        obs_buf = np.zeros((T, obs_dim), dtype=np.float32)
        act_buf = np.zeros((T,), dtype=np.int64)
        logp_buf = np.zeros((T,), dtype=np.float32)
        rew_buf = np.zeros((T,), dtype=np.float32)
        done_buf = np.zeros((T,), dtype=np.float32)
        val_buf = np.zeros((T,), dtype=np.float32)

        # Collect rollout
        for t in range(T):
            obs_buf[t] = obs
            obs_t = torch.tensor(obs, dtype=torch.float32, device=cfg_obj.device).unsqueeze(0)
            with torch.no_grad():
                logits = actor(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                value = critic(obs_t).squeeze(-1)
                action = dist.sample()
                logp = dist.log_prob(action)
            a = int(action.item())
            next_obs, reward, terminated, truncated, _ = env.step(a)

            act_buf[t] = a
            logp_buf[t] = float(logp.item())
            rew_buf[t] = float(reward)
            done = terminated or truncated
            done_buf[t] = float(done)
            val_buf[t] = float(value.item())

            obs = next_obs
            global_step += 1
            if done:
                obs, _ = env.reset()

            if global_step >= cfg_obj.total_timesteps:
                obs_buf = obs_buf[:t+1]
                act_buf = act_buf[:t+1]
                logp_buf = logp_buf[:t+1]
                rew_buf = rew_buf[:t+1]
                done_buf = done_buf[:t+1]
                val_buf = val_buf[:t+1]
                break

        # Bootstrap
        with torch.no_grad():
            last_val = critic(torch.tensor(obs, dtype=torch.float32, device=cfg_obj.device).unsqueeze(0)).item()

        # GAE
        Tlen = len(rew_buf)
        adv_buf = np.zeros_like(rew_buf)
        last_gae = 0.0
        for t in reversed(range(Tlen)):
            next_nonterminal = 1.0 - done_buf[t]
            next_value = last_val if t == Tlen - 1 else val_buf[t + 1]
            delta = rew_buf[t] + cfg_obj.gamma * next_value * next_nonterminal - val_buf[t]
            last_gae = delta + cfg_obj.gamma * cfg_obj.gae_lambda * next_nonterminal * last_gae
            adv_buf[t] = last_gae
        ret_buf = adv_buf + val_buf

        # Tensors
        adv_t = torch.tensor(adv_buf, dtype=torch.float32, device=cfg_obj.device)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std(unbiased=False) + 1e-8)
        obs_t = torch.tensor(obs_buf, dtype=torch.float32, device=cfg_obj.device)
        act_t = torch.tensor(act_buf, dtype=torch.int64, device=cfg_obj.device)
        old_logp_t = torch.tensor(logp_buf, dtype=torch.float32, device=cfg_obj.device)
        ret_t = torch.tensor(ret_buf, dtype=torch.float32, device=cfg_obj.device)

        # PPO updates with PGD projection on actor
        batch_size = Tlen
        idxs = np.arange(batch_size)
        for _ in range(cfg_obj.update_epochs):
            np.random.shuffle(idxs)
            for start in range(0, batch_size, cfg_obj.minibatch_size):
                end = start + cfg_obj.minibatch_size
                mb_idx = idxs[start:end]
                mb_obs = obs_t[mb_idx]
                mb_act = act_t[mb_idx]
                mb_old_logp = old_logp_t[mb_idx]
                mb_adv = adv_t[mb_idx]
                mb_ret = ret_t[mb_idx]

                logits = actor(mb_obs)
                dist = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(mb_act)
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - mb_old_logp)
                pg_loss1 = -mb_adv * ratio
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1.0 - cfg_obj.clip_coef, 1.0 + cfg_obj.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                v = critic(mb_obs).squeeze(-1)
                v_loss = torch.nn.functional.mse_loss(v, mb_ret)

                loss = pg_loss + cfg_obj.vf_coef * v_loss - cfg_obj.ent_coef * entropy

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(actor.parameters()) + list(critic.parameters()),
                    cfg_obj.max_grad_norm
                )
                optimizer.step()

                # Project actor back into Rashomon bounds (PGD step)
                project_actor_(actor, bounds_l, bounds_u)

        if global_step % (10 * cfg_obj.rollout_steps) < cfg_obj.rollout_steps:
            mean_r, std_r = evaluate(env_id_str, actor, torch.device(cfg_obj.device), episodes=10)
            elapsed = time.time() - start_time
            print(f"[CartPoleDiff] Steps={global_step} | meanR={mean_r:.1f} +/- {std_r:.1f} | elapsed={elapsed:.1f}s")

    env.close()
    # Final evaluation
    print(f"Actor remains nn.Sequential: {isinstance(actor, torch.nn.Sequential)}")
    mean_r, std_r = evaluate(env_id_str, actor, torch.device(cfg_obj.device), episodes=20)
    print(f"[CartPoleDiff] Final eval: mean_reward={mean_r:.2f} +/- {std_r:.2f}")
    return actor, critic

# Prepare bounds (from IntervalTrainer results)
bounds_l = [t.to(device) for t in param_bounds_l]
bounds_u = [t.to(device) for t in param_bounds_u]
assert len(bounds_l) == len(bounds_u) == sum(1 for _ in ppo_actor.parameters())

# Train on CartPoleDiff with PGD projection
ppo_actor_diff, ppo_critic_diff = ppo_train_cartpole_diff_with_pgd(
    warm_actor=ppo_actor,
    warm_critic=ppo_critic,
    bounds_l=bounds_l,
    bounds_u=bounds_u,
    cfg_obj=cfg,
    env_id_str=env_id_diff,
)

[CartPoleDiff] Steps=20480 | meanR=197.8 +/- 263.4 | elapsed=4.4s
[CartPoleDiff] Steps=40960 | meanR=28.6 +/- 11.0 | elapsed=8.9s
[CartPoleDiff] Steps=61440 | meanR=111.8 +/- 176.0 | elapsed=13.3s
[CartPoleDiff] Steps=81920 | meanR=79.1 +/- 173.7 | elapsed=17.8s
Actor remains nn.Sequential: True
[CartPoleDiff] Final eval: mean_reward=199.35 +/- 262.45


In [25]:
# Parameter projection function
def project_parameters_to_rashomon_bounds(model, param_bounds_l, param_bounds_u, device='cpu'):
    """Project model parameters to stay within Rashomon bounds."""
    total_projected = 0
    total_params = 0
    
    with torch.no_grad():
        for i, param in enumerate(model.parameters()):
            if i < len(param_bounds_l) and i < len(param_bounds_u):
                # Get bounds for this parameter
                p_l = param_bounds_l[i].view(param.shape).to(device)
                p_u = param_bounds_u[i].view(param.shape).to(device)
                
                # Count violations before projection
                violations = ((param.data < p_l) | (param.data > p_u)).sum().item()
                total_projected += violations
                total_params += param.numel()
                
                # Project parameters to bounds
                param.data.clamp_(min=p_l, max=p_u)
    
    return total_projected, total_params

In [26]:
# Verify parameter bounds compliance
print("\nVerifying Rashomon bounds compliance...")
final_projected, final_total = project_parameters_to_rashomon_bounds(
    ppo_actor_diff, param_bounds_l, param_bounds_u
)
compliance_rate = 1.0 - (final_projected / max(final_total, 1))
print(f"Parameter compliance: {compliance_rate:.4f} ({final_total - final_projected}/{final_total} parameters within bounds)")


Verifying Rashomon bounds compliance...
Parameter compliance: 1.0000 (4610/4610 parameters within bounds)


In [27]:
# Safety check of ppo_actor_diff on the original CartPole-v1 using empirical failures and the safety_critic

episodes_check = 200
max_steps_limit = getattr(gym.make(env_id).spec, "max_episode_steps", 500)

ppo_actor_diff.eval()
safety_critic.eval()

fail_episodes = 0
ep_lengths = []
mean_pred_fail_probs = []
max_pred_fail_probs = []
safe_step_count = 0
total_step_count = 0

eval_env = gym.make(env_id)

with torch.no_grad():
    for _ in range(episodes_check):
        obs, _ = eval_env.reset(seed=np.random.randint(0, 10_000_000))
        ep_probs = []
        steps = 0
        while True:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            logits = ppo_actor_diff(obs_t)
            action = torch.argmax(logits, dim=-1).item()

            # Predicted failure probability for (state, action)
            a_t = torch.tensor([[float(action)]], dtype=torch.float32, device=device)
            sa = torch.cat([obs_t, a_t], dim=1)
            pred_fail_prob = torch.sigmoid(safety_critic(sa)).item()
            ep_probs.append(pred_fail_prob)

            # Count "safe" steps per chosen action (threshold defined earlier)
            safe_step_count += 1 if pred_fail_prob < threshold else 0
            total_step_count += 1

            obs, _, terminated, truncated, info = eval_env.step(action)
            steps += 1

            if terminated or truncated or steps >= max_steps_limit:
                # Failure if terminated (not time-limit truncation)
                fail_episodes += 1 if (terminated and not truncated) else 0
                ep_lengths.append(steps)
                mean_pred_fail_probs.append(float(numpy.mean(ep_probs)) if ep_probs else 0.0)
                max_pred_fail_probs.append(float(numpy.max(ep_probs)) if ep_probs else 0.0)
                break

eval_env.close()

failure_rate = fail_episodes / episodes_check
mean_len = float(numpy.mean(ep_lengths)) if ep_lengths else 0.0
std_len = float(numpy.std(ep_lengths)) if ep_lengths else 0.0
avg_mean_pred_fail = float(numpy.mean(mean_pred_fail_probs)) if mean_pred_fail_probs else 0.0
avg_max_pred_fail = float(numpy.mean(max_pred_fail_probs)) if max_pred_fail_probs else 0.0
safe_step_ratio = (safe_step_count / max(1, total_step_count))

print("Safety check on CartPole-v1 with ppo_actor_diff")
print(f"- episodes={episodes_check}, max_steps={max_steps_limit}")
print(f"- empirical failure rate (terminated only) = {failure_rate:.4f}")
print(f"- episode length: mean={mean_len:.1f} +/- {std_len:.1f}")
print(f"- safety_critic: mean predicted fail prob per episode = {avg_mean_pred_fail:.4e}")
print(f"- safety_critic: mean of episode-wise max fail prob   = {avg_max_pred_fail:.4e}")
print(f"- fraction of steps predicted safe (p_fail < {threshold}) = {safe_step_ratio:.4f}")

Safety check on CartPole-v1 with ppo_actor_diff
- episodes=200, max_steps=500
- empirical failure rate (terminated only) = 0.0850
- episode length: mean=495.3 +/- 17.1
- safety_critic: mean predicted fail prob per episode = 9.9086e-03
- safety_critic: mean of episode-wise max fail prob   = 6.6437e-02
- fraction of steps predicted safe (p_fail < 0.5) = 0.9992


### Summary

In [29]:
# Compare safety metrics with and without Rashomon set bounds
print('Compare safety metrics between ppo_actor_diff_free and ppo_actor_diff')

def compute_safety_metrics(actor, episodes=episodes_check, env_id_str=env_id, threshold_val=threshold, device_str=device, use_safety_critic=False):
    env_local = gym.make(env_id_str)
    max_steps = getattr(env_local.spec, "max_episode_steps", 500)

    actor.eval()
    if use_safety_critic:
        safety_critic.eval()

    fail_eps = 0
    ep_lengths_local = []
    mean_pred_list = []
    max_pred_list = []
    safe_steps = 0
    total_steps = 0

    with torch.no_grad():
        for _ in range(episodes):
            obs_local, _ = env_local.reset(seed=np.random.randint(0, 10_000_000))
            ep_probs_local = []
            steps_local = 0
            while True:
                obs_t_local = torch.tensor(obs_local, dtype=torch.float32, device=device_str).unsqueeze(0)
                logits_local = actor(obs_t_local)
                a_local = torch.argmax(logits_local, dim=-1).item()

                a_t_local = torch.tensor([[float(a_local)]], dtype=torch.float32, device=device_str)
                sa_local = torch.cat([obs_t_local, a_t_local], dim=1)
                if use_safety_critic:
                    p_fail_local = torch.sigmoid(safety_critic(sa_local)).item()
                    ep_probs_local.append(p_fail_local)
                    safe_steps += 1 if p_fail_local < threshold_val else 0
                total_steps += 1

                obs_local, _, terminated_local, truncated_local, _ = env_local.step(a_local)
                done_local = terminated_local or truncated_local
                steps_local += 1

                if not use_safety_critic:
                    # NOTE: failure condition only for CartPole-v1
                    safe_steps += 1 if not (terminated_local and not truncated_local) else 0

                if done_local:
                    if terminated_local and not truncated_local:
                        fail_eps += 1
                    ep_lengths_local.append(steps_local)
                    if use_safety_critic:
                        mean_pred_list.append(float(np.mean(ep_probs_local)) if ep_probs_local else 0.0)
                        max_pred_list.append(float(np.max(ep_probs_local)) if ep_probs_local else 0.0)
                    break

    env_local.close()

    return {
        "episodes": episodes,
        "failure_rate": fail_eps / episodes,
        "mean_ep_len": float(np.mean(ep_lengths_local)) if ep_lengths_local else 0.0,
        "std_ep_len": float(np.std(ep_lengths_local)) if ep_lengths_local else 0.0,
        "avg_mean_pred_fail": float(np.mean(mean_pred_list)) if mean_pred_list else 0.0,
        "avg_max_pred_fail": float(np.mean(max_pred_list)) if max_pred_list else 0.0,
        "safe_step_ratio": safe_steps / max(1, total_steps),
    }

print('Using safety critic?', use_safety_critic := False)
metrics_free = compute_safety_metrics(ppo_actor_diff_free, env_id_str='CartPole-v1', use_safety_critic=use_safety_critic)
metrics_bound = compute_safety_metrics(ppo_actor_diff, env_id_str='CartPole-v1', use_safety_critic=use_safety_critic)

# Tabulate comparison
comp_df = pd.DataFrame([metrics_free, metrics_bound], index=["No bounds", "With bounds"])
diff_series = comp_df.loc["With bounds"] - comp_df.loc["No bounds"]

print("Safety metrics comparison (CartPole-v1 using safety_critic):")
print(comp_df.to_string(float_format=lambda x: f"{x:.6f}"))
print("\nDelta (With bounds - No bounds):")
print(diff_series.to_string(float_format=lambda x: f"{x:.6f}"))

Compare safety metrics between ppo_actor_diff_free and ppo_actor_diff
Using safety critic? False
Safety metrics comparison (CartPole-v1 using safety_critic):
             episodes  failure_rate  mean_ep_len  std_ep_len  avg_mean_pred_fail  avg_max_pred_fail  safe_step_ratio
No bounds         200      0.110000   492.815000   22.277360            0.000000           0.000000         0.999777
With bounds       200      0.105000   495.105000   17.164614            0.000000           0.000000         0.999788

Delta (With bounds - No bounds):
episodes              0.000000
failure_rate         -0.005000
mean_ep_len           2.290000
std_ep_len           -5.112746
avg_mean_pred_fail    0.000000
avg_max_pred_fail     0.000000
safe_step_ratio       0.000011
